# Clustering Experiment (Version Colab)

Esta version esta adaptada para Google Colab:
- Gestiona imports locales del repositorio automaticamente.
- Sustituye Ollama por embeddings de Hugging Face (sin despliegue local).
- Mantiene la estructura del experimento original.

In [1]:
# Instalacion de dependencias en Colab
%pip -q install datasets sentence-transformers umap-learn seaborn

In [2]:
# Clonar o reutilizar el repositorio en /content
from pathlib import Path
import os

REPO_DIR = Path('/content/TFG-Chatbot')
if not REPO_DIR.exists():
    !git clone https://github.com/GabrielFranciscoSM/TFG-Chatbot.git /content/TFG-Chatbot

%cd /content/TFG-Chatbot
print(f'Repo listo en: {REPO_DIR}')

/content/TFG-Chatbot
Repo listo en: /content/TFG-Chatbot


In [3]:
# Imports, carga de modulos locales y configuracion global
from __future__ import annotations

import importlib.util
import json
import logging
import pickle
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

def _detect_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, Path('/content/TFG-Chatbot')]
    for c in candidates:
        if (c / 'validation' / 'clustering.py').exists():
            return c
    raise FileNotFoundError('No se encontro el root del repo con validation/clustering.py')

ROOT = _detect_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def _load_local_module(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    assert spec is not None and spec.loader is not None
    spec.loader.exec_module(module)
    return module

clustering_module = _load_local_module('tfg_validation_clustering', ROOT / 'validation' / 'clustering.py')
datasets_module = _load_local_module('tfg_validation_datasets', ROOT / 'validation' / 'datasets.py')
metrics_module = _load_local_module('tfg_validation_metrics', ROOT / 'validation' / 'metrics' / 'metrics.py')

GenericKMeans = clustering_module.GenericKMeans
DatasetLoader = datasets_module.DatasetLoader
evaluate_fuzzy_clustering = metrics_module.evaluate_fuzzy_clustering
evaluate_hard_clustering = metrics_module.evaluate_hard_clustering

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print(f'ROOT detectado: {ROOT}')
print(f'UMAP disponible: {HAS_UMAP}')

ROOT detectado: /content/TFG-Chatbot
UMAP disponible: True


In [4]:
# Directorios de salida y logging
RESULTS_DIR = ROOT / 'validation' / 'results' / 'clustering_experiment_colab'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
ARTIFACTS_DIR = RESULTS_DIR / 'artifacts'
LOGS_DIR = RESULTS_DIR / 'logs'

for p in [RESULTS_DIR, TABLES_DIR, FIGURES_DIR, ARTIFACTS_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
    handlers=[
        logging.FileHandler(LOGS_DIR / 'experiment.log'),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger('clustering_experiment_colab')

def save_table(df: pd.DataFrame, filename: str) -> Path:
    path = TABLES_DIR / filename
    df.to_csv(path, index=False)
    return path

def save_figure(fig: plt.Figure, filename: str, dpi: int = 180) -> Path:
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    return path

print(f'Resultados en: {RESULTS_DIR}')

Resultados en: /content/TFG-Chatbot/validation/results/clustering_experiment_colab


In [5]:
# Carga y limpieza de datasets
from datasets import load_dataset

loader = DatasetLoader()

PILOT_MODE = True
PILOT_LIMITS = {
    'dialogsum': 1200,
    'stackoverflow': 1200,
    'esquad': 1200,
}

def _standardize_stackoverflow_df(df: pd.DataFrame) -> pd.DataFrame:
    text_col = next(
        (
            c
            for c in [
                'func_documentation_string',
                'docstring',
                'whole_func_string',
                'func_code_string',
                'question',
                'title',
                'text',
                'body',
            ]
            if c in df.columns
        ),
        None,
    )
    if text_col is None:
        raise ValueError(f'No text-like column found in StackOverflow fallback dataset: {list(df.columns)}')

    out = (
        df[[text_col]]
        .rename(columns={text_col: 'text'})
        .assign(label='Technical')
    )
    out = out.dropna(subset=['text'])
    out = out[out['text'].astype(str).str.strip().str.len() > 0]
    return out.reset_index(drop=True)

def _standardize_esquad_df(df: pd.DataFrame) -> pd.DataFrame:
    text_col = next((c for c in ['question', 'text', 'context'] if c in df.columns), None)
    if text_col is None:
        raise ValueError(f'No text-like column found in Spanish QA fallback dataset: {list(df.columns)}')

    out = (
        df[[text_col]]
        .rename(columns={text_col: 'text'})
        .assign(label='Education_ES')
    )
    out = out.dropna(subset=['text'])
    out = out[out['text'].astype(str).str.strip().str.len() > 0]
    return out.reset_index(drop=True)

def load_stackoverflow_colab(limit: int = 5000) -> pd.DataFrame:
    errors = []

    candidates = [
        ('code_search_net', 'python', 'train'),
        ('SetFit/stackoverflow_questions', None, 'train'),
    ]

    for ds_name, ds_config, split in candidates:
        try:
            if ds_config is None:
                ds = load_dataset(ds_name, split=split)
            else:
                ds = load_dataset(ds_name, ds_config, split=split)
            df = pd.DataFrame(ds).head(limit)
            return _standardize_stackoverflow_df(df)
        except Exception as e:
            errors.append(f'{ds_name}: {e}')

    raise RuntimeError(
        'No se pudo cargar un dataset de StackOverflow compatible en Colab sin trust_remote_code. '\
        f'Errores: {errors}'
    )

def load_esquad_colab(limit: int | None = None) -> pd.DataFrame:
    errors = []

    # xquad.es es estable y no depende de loading scripts con trust_remote_code
    candidates = [
        ('google/xquad', 'xquad.es', 'validation'),
        ('PlanTL-GOB-ES/SQAC', None, 'train'),
    ]

    for ds_name, ds_config, split in candidates:
        try:
            if ds_config is None:
                ds = load_dataset(ds_name, split=split)
            else:
                ds = load_dataset(ds_name, ds_config, split=split)
            df = pd.DataFrame(ds)
            if limit is not None:
                df = df.head(limit)
            return _standardize_esquad_df(df)
        except Exception as e:
            errors.append(f'{ds_name}: {e}')

    raise RuntimeError(
        'No se pudo cargar un dataset ES en Colab sin trust_remote_code. '\
        f'Errores: {errors}'
    )

df_dialogsum = loader.load_dialogsum()
df_stack = load_stackoverflow_colab(limit=PILOT_LIMITS['stackoverflow'] if PILOT_MODE else 5000)
df_esquad = load_esquad_colab(limit=PILOT_LIMITS['esquad'] if PILOT_MODE else None)

if PILOT_MODE:
    df_dialogsum = df_dialogsum.head(PILOT_LIMITS['dialogsum'])

for df, name in [
    (df_dialogsum, 'dialogsum'),
    (df_stack, 'stackoverflow'),
    (df_esquad, 'esquad'),
]:
    df['dataset'] = name

datasets_raw = {
    'dialogsum': df_dialogsum[['text', 'label', 'dataset']].copy(),
    'stackoverflow': df_stack[['text', 'label', 'dataset']].copy(),
    'esquad': df_esquad[['text', 'label', 'dataset']].copy(),
}

def clean_dataset(df: pd.DataFrame, min_len: int = 15) -> pd.DataFrame:
    out = df.copy()
    out = out.dropna(subset=['text'])
    out['text'] = out['text'].astype(str).str.strip()
    out = out[out['text'].str.len() > 0]
    out = out.drop_duplicates(subset=['text'])
    out = out[out['text'].str.len() >= min_len]
    return out.reset_index(drop=True)

datasets = {name: clean_dataset(df, min_len=15) for name, df in datasets_raw.items()}

df_sample_report = pd.DataFrame([
    {
        'dataset': name,
        'n_docs': len(df),
        'avg_text_len': float(df['text'].str.len().mean()),
        'n_unique_labels': int(df['label'].nunique()),
    }
    for name, df in datasets.items()
]).sort_values('dataset')
save_table(df_sample_report, 'sample_report.csv')
df_sample_report

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'knkarthick/dialogsum' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'knkarthick/dialogsum' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading DialogSum...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

xquad.es/validation-00000-of-00001.parqu(…):   0%|          | 0.00/237k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1190 [00:00<?, ? examples/s]

,dataset,n_docs,avg_text_len,n_unique_labels
0,dialogsum,1200,740.635833,1006
2,esquad,1184,68.238176,1
1,stackoverflow,1125,199.427556,1


In [6]:
# Representaciones: BoW, TF-IDF y embeddings HF
def build_bow(texts: list[str], params: dict):
    start = time.perf_counter()
    vec = CountVectorizer(**params)
    X = vec.fit_transform(texts).astype(np.float64)
    return X, vec, time.perf_counter() - start

def build_tfidf(texts: list[str], params: dict):
    start = time.perf_counter()
    vec = TfidfVectorizer(**params)
    X = vec.fit_transform(texts).astype(np.float64)
    return X, vec, time.perf_counter() - start

def build_hf_embeddings(texts: list[str], batch_size: int, model_name: str):
    start = time.perf_counter()
    model = SentenceTransformer(model_name)
    X = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    X = np.asarray(X, dtype=np.float32)
    return X, model, time.perf_counter() - start

REP_CONFIG = {
    'bow': {
        'max_features': 1200 if PILOT_MODE else 3000,
        'ngram_range': (1, 2),
        'min_df': 2,
        'max_df': 0.95,
    },
    'tfidf': {
        'max_features': 1200 if PILOT_MODE else 3000,
        'ngram_range': (1, 2),
        'min_df': 2,
        'max_df': 0.95,
        'sublinear_tf': True,
    },
    'hf_embed': {
        'model_name': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
        'batch_size': 64 if PILOT_MODE else 128,
    },
}

representations = {}
representation_meta = []

for ds_name, df in datasets.items():
    texts = df['text'].tolist()
    representations[ds_name] = {}

    bow_cache = ARTIFACTS_DIR / f'{ds_name}_bow.pkl'
    if bow_cache.exists():
        payload = pickle.loads(bow_cache.read_bytes())
        X_bow = payload['X']
        bow_elapsed = payload.get('elapsed_sec', np.nan)
    else:
        X_bow, bow_vec, bow_elapsed = build_bow(texts, REP_CONFIG['bow'])
        X_bow = normalize(X_bow, norm='l2', axis=1)
        bow_cache.write_bytes(pickle.dumps({'X': X_bow, 'vectorizer': bow_vec, 'elapsed_sec': bow_elapsed}))
    representations[ds_name]['bow'] = X_bow
    representation_meta.append({'dataset': ds_name, 'representation': 'bow', 'shape': str(X_bow.shape), 'build_sec': bow_elapsed})

    tfidf_cache = ARTIFACTS_DIR / f'{ds_name}_tfidf.pkl'
    if tfidf_cache.exists():
        payload = pickle.loads(tfidf_cache.read_bytes())
        X_tfidf = payload['X']
        tfidf_elapsed = payload.get('elapsed_sec', np.nan)
    else:
        X_tfidf, tfidf_vec, tfidf_elapsed = build_tfidf(texts, REP_CONFIG['tfidf'])
        X_tfidf = normalize(X_tfidf, norm='l2', axis=1)
        tfidf_cache.write_bytes(pickle.dumps({'X': X_tfidf, 'vectorizer': tfidf_vec, 'elapsed_sec': tfidf_elapsed}))
    representations[ds_name]['tfidf'] = X_tfidf
    representation_meta.append({'dataset': ds_name, 'representation': 'tfidf', 'shape': str(X_tfidf.shape), 'build_sec': tfidf_elapsed})

    hf_cache = ARTIFACTS_DIR / f'{ds_name}_hf_embed.npy'
    hf_meta = ARTIFACTS_DIR / f'{ds_name}_hf_embed_meta.json'
    if hf_cache.exists() and hf_meta.exists():
        X_hf = np.load(hf_cache)
        hf_elapsed = json.loads(hf_meta.read_text()).get('elapsed_sec', np.nan)
    else:
        X_hf, _, hf_elapsed = build_hf_embeddings(
            texts=texts,
            batch_size=REP_CONFIG['hf_embed']['batch_size'],
            model_name=REP_CONFIG['hf_embed']['model_name'],
        )
        np.save(hf_cache, X_hf)
        hf_meta.write_text(json.dumps({'elapsed_sec': hf_elapsed}))
    representations[ds_name]['hf_embed'] = X_hf
    representation_meta.append({'dataset': ds_name, 'representation': 'hf_embed', 'shape': str(X_hf.shape), 'build_sec': hf_elapsed})

df_rep_meta = pd.DataFrame(representation_meta)
save_table(df_rep_meta, 'representation_build_report.csv')
df_rep_meta

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

,dataset,representation,shape,build_sec
0,dialogsum,bow,"(1200, 1200)",0.328803
1,dialogsum,tfidf,"(1200, 1200)",0.330044
2,dialogsum,hf_embed,"(1200, 384)",7.850448
3,stackoverflow,bow,"(1125, 1200)",0.075105
4,stackoverflow,tfidf,"(1125, 1200)",0.072360
5,stackoverflow,hf_embed,"(1125, 384)",3.909028
6,esquad,bow,"(1184, 1200)",0.036441
7,esquad,tfidf,"(1184, 1200)",0.035615
8,esquad,hf_embed,"(1184, 384)",3.952773


In [7]:
# Grid, runner y ejecucion completa
K_VALUES = list(range(2, 8)) if PILOT_MODE else list(range(2, 16))
SEEDS = [42, 52, 62] if PILOT_MODE else [42, 52, 62, 72, 82]

ALGO_CONFIGS = [
    {'algorithm': 'kmeans', 'distance': 'euclidean', 'm': None},
    {'algorithm': 'kmeans', 'distance': 'cosine', 'm': None},
    {'algorithm': 'fcm', 'distance': 'euclidean', 'm': 2.0},
    {'algorithm': 'fcm', 'distance': 'cosine', 'm': 2.0},
]

rows = []
for ds_name, rep_dict in representations.items():
    for rep_name, X in rep_dict.items():
        if X is None:
            continue
        n = X.shape[0]
        k_values_ds = [k for k in K_VALUES if k < n]
        for cfg in ALGO_CONFIGS:
            for k in k_values_ds:
                for seed in SEEDS:
                    rows.append({
                        'dataset': ds_name,
                        'representation': rep_name,
                        'algorithm': cfg['algorithm'],
                        'distance': cfg['distance'],
                        'm': cfg['m'],
                        'k': k,
                        'seed': seed,
                    })

df_grid = pd.DataFrame(rows)
save_table(df_grid, 'experiment_grid.csv')

def _to_dense_if_needed(X):
    return X.toarray() if hasattr(X, 'toarray') else X

def run_single_experiment(X, dataset_name, representation_name, algorithm, distance, k, seed, m=None, max_iter=200, tol=1e-4):
    X_arr = _to_dense_if_needed(X)

    model = GenericKMeans(
        n_clusters=k,
        algorithm=algorithm,
        distance=distance,
        max_iter=max_iter,
        tol=tol,
        random_state=seed,
        m=2.0 if m is None else m,
    )

    start = time.perf_counter()
    model.fit(X_arr)
    elapsed = time.perf_counter() - start

    if algorithm == 'fcm':
        metrics = evaluate_fuzzy_clustering(
            X_arr,
            model.membership_,
            model.centroids_,
            m=2.0 if m is None else m,
            distance=distance,
        )
    else:
        metrics = evaluate_hard_clustering(X_arr, model.labels_, metric=distance)

    return {
        'dataset': dataset_name,
        'representation': representation_name,
        'algorithm': algorithm,
        'distance': distance,
        'm': m,
        'k': k,
        'seed': seed,
        'asw': metrics.asw,
        'ch': metrics.ch,
        'pc': metrics.pc,
        'pe': metrics.pe,
        'xb': metrics.xb,
        'inertia': model.inertia_,
        'n_iter': model.n_iter_,
        'runtime_sec': elapsed,
    }

results, errors = [], []
for _, row in df_grid.iterrows():
    ds, rep = row['dataset'], row['representation']
    X = representations[ds][rep]
    try:
        out = run_single_experiment(
            X=X,
            dataset_name=ds,
            representation_name=rep,
            algorithm=row['algorithm'],
            distance=row['distance'],
            k=int(row['k']),
            seed=int(row['seed']),
            m=row['m'],
        )
        results.append(out)
    except Exception as e:
        errors.append({
            'dataset': ds,
            'representation': rep,
            'algorithm': row['algorithm'],
            'distance': row['distance'],
            'k': int(row['k']),
            'seed': int(row['seed']),
            'error': str(e),
        })

df_results = pd.DataFrame(results)
df_errors = pd.DataFrame(errors)
if not df_results.empty:
    save_table(df_results, 'raw_results.csv')
if not df_errors.empty:
    save_table(df_errors, 'execution_errors.csv')

df_results.head(), len(df_results), len(df_errors)

(     dataset representation algorithm   distance   m  k  seed       asw  \
 0  dialogsum            bow    kmeans  euclidean NaN  2    42  0.025764   
 1  dialogsum            bow    kmeans  euclidean NaN  2    52  0.025601   
 2  dialogsum            bow    kmeans  euclidean NaN  2    62  0.025516   
 3  dialogsum            bow    kmeans  euclidean NaN  3    42  0.021109   
 4  dialogsum            bow    kmeans  euclidean NaN  3    52  0.022236   
 
           ch  pc  pe  xb     inertia  n_iter  runtime_sec  
 0  39.707823 NaN NaN NaN  814.424805      14     0.428159  
 1  39.717680 NaN NaN NaN  814.418318      12     0.357204  
 2  39.719913 NaN NaN NaN  814.416849      17     0.497849  
 3  30.535877 NaN NaN NaN  800.573202      17     0.665446  
 4  30.582112 NaN NaN NaN  800.514364      17     0.654447  ,
 648,
 0)

In [ ]:
# Agregacion, ranking y seleccion de mejores configuraciones
def ci95(series: pd.Series) -> float:
    s = series.dropna()
    if len(s) <= 1:
        return np.nan
    return 1.96 * float(s.std(ddof=1)) / np.sqrt(len(s))

group_cols = ['dataset', 'representation', 'algorithm', 'distance', 'k']
metric_cols = ['asw', 'ch', 'pc', 'pe', 'xb', 'runtime_sec', 'n_iter']

agg_dict = {}
for mcol in metric_cols:
    agg_dict[f'{mcol}_mean'] = (mcol, 'mean')
    agg_dict[f'{mcol}_std'] = (mcol, 'std')
    agg_dict[f'{mcol}_ci95'] = (mcol, ci95)

df_summary = df_results.groupby(group_cols).agg(**agg_dict).reset_index()
save_table(df_summary, 'summary_by_condition.csv')

df_rank = df_summary.copy()
df_rank['rank_asw'] = df_rank.groupby(['dataset', 'representation'])['asw_mean'].rank(ascending=False, method='min')
df_rank['rank_runtime'] = df_rank.groupby(['dataset', 'representation'])['runtime_sec_mean'].rank(ascending=True, method='min')
df_rank['rank_n_iter'] = df_rank.groupby(['dataset', 'representation'])['n_iter_mean'].rank(ascending=True, method='min')

df_rank = df_rank.sort_values(
    ['dataset', 'representation', 'asw_mean', 'runtime_sec_mean', 'n_iter_mean'],
    ascending=[True, True, False, True, True],
)

df_best_per_condition = df_rank.groupby(['dataset', 'representation'], as_index=False, sort=False).first()
df_best_global = df_rank.sort_values(['asw_mean', 'runtime_sec_mean', 'n_iter_mean'], ascending=[False, True, True]).head(20)

save_table(df_rank, 'summary_ranked_full.csv')
save_table(df_best_per_condition, 'summary_best_per_condition.csv')
save_table(df_best_global, 'summary_best_global_top20.csv')

display(df_best_per_condition)
df_best_global.head(10)

In [ ]:
# Visualizaciones principales
curve_metrics = ['asw_mean', 'ch_mean', 'xb_mean', 'pc_mean', 'pe_mean']

for (ds, rep), grp in df_summary.groupby(['dataset', 'representation']):
    fig, axes = plt.subplots(1, len(curve_metrics), figsize=(5 * len(curve_metrics), 4), sharex=True)
    for ax, metric in zip(axes, curve_metrics):
        plot_df = grp.copy()
        plot_df['variant'] = plot_df['algorithm'] + '_' + plot_df['distance']
        sns.lineplot(data=plot_df, x='k', y=metric, hue='variant', marker='o', ax=ax)
        ax.set_title(f'{metric} vs k')
        ax.legend(loc='best', fontsize=8)

    fig.suptitle(f'Metricas vs k | {ds} | {rep}', y=1.02)
    save_figure(fig, f'curves_{ds}_{rep}.png')
    plt.close(fig)

df_scatter = df_summary.copy()
df_scatter['algorithm_distance'] = df_scatter['algorithm'] + '_' + df_scatter['distance']
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=df_scatter,
    x='runtime_sec_mean',
    y='asw_mean',
    hue='representation',
    style='algorithm_distance',
    s=90,
    alpha=0.85,
    ax=ax,
)
ax.set_title('Trade-off global calidad-coste (ASW vs runtime)')
ax.set_xlabel('Runtime medio (s)')
ax.set_ylabel('ASW medio')
ax.legend(loc='best', fontsize=8)
save_figure(fig, 'scatter_quality_vs_cost.png')
plt.close(fig)

print('Figuras guardadas.')

## Nota para ejecucion completa

1. Cambia `PILOT_MODE = False` para el barrido final.
2. Si Colab se queda sin memoria, reduce `PILOT_LIMITS` o `max_features`.
3. Esta version usa `hf_embed` (Hugging Face) en vez de `nomic`/Ollama.